# Análisis de Sensibilidad

El experimento solo considera el uso del modelo Árbol de decisión al ser el más rápido a nivel computacional

In [12]:
import time
import threading
import os
import gc
from pathlib import Path

import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import psutil

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.base import clone


# =========================================================
# CONFIG
# =========================================================
RUTA_DATASET = "../1_data_processed/v3_dataset_post_chi.parquet"
RUTA_SPLIT = Path("../1_data_processed/split_indices")


N_SAMPLE_VALUES = [10, 50, 100, 150, 200, 500, 1000]
N_ITER = 100

K_FOLDS = 5
BASE_SEED = 42
PROGRESS_EVERY = 25

CARPETA_OUT = Path("../4_results/modelo_dt_sensibilidad_n")
CARPETA_OUT.mkdir(parents=True, exist_ok=True)

MODELO = DecisionTreeClassifier(
    criterion="gini",
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=BASE_SEED,
    class_weight="balanced"
)


# =========================================================
# MONITOR DE MEMORIA
# =========================================================
class MemoryMonitor:
    """
    Monitorea memoria RSS del proceso en MB.
    No mide peak absoluto del sistema, sino peak observado del proceso Python.
    """
    def __init__(self, interval=0.25):
        self.interval = interval
        self.process = psutil.Process(os.getpid())
        self.peak_mb = 0.0
        self.running = False
        self.thread = None

    def _monitor(self):
        while self.running:
            rss_mb = self.process.memory_info().rss / (1024 ** 2)
            if rss_mb > self.peak_mb:
                self.peak_mb = rss_mb
            time.sleep(self.interval)

    def start(self):
        self.running = True
        self.peak_mb = self.process.memory_info().rss / (1024 ** 2)
        self.thread = threading.Thread(target=self._monitor, daemon=True)
        self.thread.start()

    def stop(self):
        self.running = False
        if self.thread is not None:
            self.thread.join()

    def current_mb(self):
        return self.process.memory_info().rss / (1024 ** 2)


# =========================================================
# HELPERS
# =========================================================
def evaluar_modelo(modelo, X_eval, y_eval):
    y_pred = modelo.predict(X_eval)
    tn, fp, fn, tp = confusion_matrix(y_eval, y_pred, labels=[0, 1]).ravel()

    acc = accuracy_score(y_eval, y_pred)
    prec = precision_score(y_eval, y_pred, zero_division=0)
    rec = recall_score(y_eval, y_pred, zero_division=0)
    f1 = f1_score(y_eval, y_pred, zero_division=0)

    roc = np.nan
    pr = np.nan

    try:
        if hasattr(modelo, "predict_proba"):
            y_proba = modelo.predict_proba(X_eval)[:, 1]
            roc = roc_auc_score(y_eval, y_proba)
            pr = average_precision_score(y_eval, y_proba)
    except Exception:
        pass

    return acc, prec, rec, f1, roc, pr, tp, fp, tn, fn


def preparar_pyarrow():
    for ext_name in ["pandas.period", "pandas.interval"]:
        try:
            pa.unregister_extension_type(ext_name)
        except Exception:
            pass


def compactar_tipos(df):
    for c in df.columns:
        if c.startswith("signo_zodiacal_"):
            df[c] = df[c].astype("uint8")
        elif c == "ESTANCIA_DIAS":
            df[c] = df[c].astype("int32")
        elif pd.api.types.is_numeric_dtype(df[c]):
            df[c] = df[c].astype("uint8")
    return df


def validar_n_sample(n_sample, k_folds):
    if n_sample < 2:
        raise ValueError("N_SAMPLE debe ser al menos 2.")

    n_pos = n_sample // 2
    n_neg = n_sample - n_pos

    if n_pos < k_folds or n_neg < k_folds:
        print(
            f"Advertencia: n_sample={n_sample} genera "
            f"{n_pos} positivos y {n_neg} negativos. "
            f"Con k={k_folds}, cada fold tendrá muy pocos casos por clase."
        )

    return n_pos, n_neg


# =========================================================
# CARGA DE DATOS
# =========================================================
preparar_pyarrow()

cols = pq.read_schema(RUTA_DATASET).names
cols_signo = [c for c in cols if c.startswith("signo_zodiacal_")]

if not cols_signo:
    raise ValueError("No se encontraron columnas signo_zodiacal_*.")

df = pd.read_parquet(RUTA_DATASET, engine="pyarrow")
print("Dataset cargado:", df.shape)

df = compactar_tipos(df)

idx_train = np.load(RUTA_SPLIT / "idx_train.npy")

X_all = df.drop(columns=cols_signo).to_numpy(copy=False)
df_train = df.iloc[idx_train].reset_index(drop=True)
X_train_all = X_all[idx_train]

skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=BASE_SEED)

print("Columnas signo:", len(cols_signo))
print("Tamaños a evaluar:", N_SAMPLE_VALUES)
print("Iteraciones por tamaño:", N_ITER)


# =========================================================
# RUN SENSIBILIDAD
# =========================================================
resultados_globales = []
ejecucion_global = []

monitor = MemoryMonitor(interval=0.25)

for n_sample in N_SAMPLE_VALUES:
    print(f"\n=================================================")
    print(f"DT sensibilidad -> N_SAMPLE = {n_sample}")
    print(f"=================================================")

    n_pos, n_neg = validar_n_sample(n_sample, K_FOLDS)

    t_inicio_n = time.perf_counter()
    monitor.start()
    memoria_inicio_mb = monitor.current_mb()

    resultados_n = []

    for signo in cols_signo:
        print(f"\nDT -> {signo} | n={n_sample}")

        y_train_all = df_train[signo].to_numpy(dtype=np.uint8)

        pos_idx = np.where(y_train_all == 1)[0]
        neg_idx = np.where(y_train_all == 0)[0]

        if len(pos_idx) == 0 or len(neg_idx) == 0:
            print(f"Saltando {signo}")
            continue

        resultados_signo = []

        for it in range(N_ITER):
            seed_signo = sum(ord(ch) for ch in signo)
            seed = BASE_SEED + it + seed_signo + n_sample
            rng = np.random.default_rng(seed)

            sample_pos = rng.choice(pos_idx, size=n_pos, replace=True)
            sample_neg = rng.choice(neg_idx, size=n_neg, replace=True)

            sample_idx = np.concatenate([sample_pos, sample_neg])
            rng.shuffle(sample_idx)

            X_boot = X_train_all[sample_idx]
            y_boot = y_train_all[sample_idx]

            fold_metrics = []

            for tr_idx, val_idx in skf.split(X_boot, y_boot):
                X_tr, X_val = X_boot[tr_idx], X_boot[val_idx]
                y_tr, y_val = y_boot[tr_idx], y_boot[val_idx]

                m_cv = clone(MODELO)
                m_cv.fit(X_tr, y_tr)

                fold_metrics.append(evaluar_modelo(m_cv, X_val, y_val))

            fold_metrics = np.array(fold_metrics, dtype=float)

            resultados_signo.append({
                "modelo": "dt",
                "n_sample": n_sample,
                "n_pos": n_pos,
                "n_neg": n_neg,
                "signo": signo,
                "iter": it + 1,
                "pos_rate_bootstrap": float(y_boot.mean()),
                "cv_accuracy": np.nanmean(fold_metrics[:, 0]),
                "cv_precision": np.nanmean(fold_metrics[:, 1]),
                "cv_recall": np.nanmean(fold_metrics[:, 2]),
                "cv_f1": np.nanmean(fold_metrics[:, 3]),
                "cv_roc_auc": np.nanmean(fold_metrics[:, 4]),
                "cv_pr_auc": np.nanmean(fold_metrics[:, 5]),
                "cv_tp_mean": np.nanmean(fold_metrics[:, 6]),
                "cv_fp_mean": np.nanmean(fold_metrics[:, 7]),
                "cv_tn_mean": np.nanmean(fold_metrics[:, 8]),
                "cv_fn_mean": np.nanmean(fold_metrics[:, 9]),
            })

            if (it + 1) % PROGRESS_EVERY == 0:
                print(f"   Iter {it + 1}/{N_ITER}")

        df_signo = pd.DataFrame(resultados_signo)

        # Guardado parcial por signo y tamaño
        nombre_signo = signo.replace("signo_zodiacal_", "")
        path_signo = CARPETA_OUT / f"dt_detalle_n{n_sample}_{nombre_signo}.csv"
        df_signo.to_csv(path_signo, index=False)

        resultados_n.extend(resultados_signo)
        resultados_globales.extend(resultados_signo)

        del resultados_signo, df_signo
        gc.collect()

    monitor.stop()
    t_fin_n = time.perf_counter()

    memoria_fin_mb = monitor.current_mb()
    tiempo_total_seg = t_fin_n - t_inicio_n

    total_iteraciones = len(cols_signo) * N_ITER
    total_cv_fits = total_iteraciones * K_FOLDS

    ejecucion_global.append({
        "modelo": "dt",
        "n_sample": n_sample,
        "n_iter": N_ITER,
        "k_folds": K_FOLDS,
        "n_signos": len(cols_signo),
        "total_iteraciones": total_iteraciones,
        "total_cv_fits": total_cv_fits,
        "tiempo_total_seg": tiempo_total_seg,
        "tiempo_promedio_iteracion_seg": tiempo_total_seg / total_iteraciones,
        "tiempo_promedio_fit_cv_seg": tiempo_total_seg / total_cv_fits,
        "memoria_inicio_mb": memoria_inicio_mb,
        "memoria_fin_mb": memoria_fin_mb,
        "memoria_peak_observada_mb": monitor.peak_mb,
    })

    df_n = pd.DataFrame(resultados_n)
    df_n.to_csv(CARPETA_OUT / f"dt_detalle_n{n_sample}.csv", index=False)

    print(f"\nTerminado n={n_sample}")
    print(f"Tiempo total: {tiempo_total_seg:.2f} segundos")
    print(f"Memoria peak observada: {monitor.peak_mb:.2f} MB")

    del resultados_n, df_n
    gc.collect()


# =========================================================
# GUARDADO FINAL
# =========================================================
df_detalle = pd.DataFrame(resultados_globales)
detalle_path = CARPETA_OUT / "dt_sensibilidad_detalle.csv"
df_detalle.to_csv(detalle_path, index=False)

cols_metricas = [
    "pos_rate_bootstrap",
    "cv_accuracy", "cv_precision", "cv_recall", "cv_f1", "cv_roc_auc", "cv_pr_auc",
    "cv_tp_mean", "cv_fp_mean", "cv_tn_mean", "cv_fn_mean"
]

if not df_detalle.empty:
    # Resumen por n_sample y signo
    df_resumen_signo = (
        df_detalle
        .groupby(["n_sample", "signo"])[cols_metricas]
        .agg(["mean", "std", "min", "max"])
    )
    resumen_signo_path = CARPETA_OUT / "dt_sensibilidad_resumen_por_signo.csv"
    df_resumen_signo.to_csv(resumen_signo_path)

    # Resumen global por n_sample
    df_resumen_global = (
        df_detalle
        .groupby("n_sample")[cols_metricas]
        .agg(["mean", "std", "min", "max"])
    )

    # Aplanar nombres de columnas
    df_resumen_global.columns = [
        f"{metrica}_{stat}" for metrica, stat in df_resumen_global.columns
    ]
    df_resumen_global = df_resumen_global.reset_index()

    resumen_global_path = CARPETA_OUT / "dt_sensibilidad_resumen_global.csv"
    df_resumen_global.to_csv(resumen_global_path, index=False)

    # Métricas de ejecución
    df_ejecucion = pd.DataFrame(ejecucion_global)
    ejecucion_path = CARPETA_OUT / "dt_sensibilidad_ejecucion.csv"
    df_ejecucion.to_csv(ejecucion_path, index=False)

    print("\n======================================")
    print("DT sensibilidad terminado")
    print("Detalle:", detalle_path)
    print("Resumen por signo:", resumen_signo_path)
    print("Resumen global:", resumen_global_path)
    print("Ejecución:", ejecucion_path)
    print("======================================")

else:
    print("No hubo resultados para guardar.")

Dataset cargado: (5808498, 488)
Columnas signo: 12
Tamaños a evaluar: [10, 50, 100, 150, 200, 500, 1000]
Iteraciones por tamaño: 100

DT sensibilidad -> N_SAMPLE = 10

DT -> signo_zodiacal_acuario | n=10
   Iter 25/100
   Iter 50/100
   Iter 75/100
   Iter 100/100

DT -> signo_zodiacal_aries | n=10
   Iter 25/100
   Iter 50/100
   Iter 75/100
   Iter 100/100

DT -> signo_zodiacal_capricornio | n=10
   Iter 25/100
   Iter 50/100
   Iter 75/100
   Iter 100/100

DT -> signo_zodiacal_cancer | n=10
   Iter 25/100
   Iter 50/100
   Iter 75/100
   Iter 100/100

DT -> signo_zodiacal_escorpio | n=10
   Iter 25/100
   Iter 50/100
   Iter 75/100
   Iter 100/100

DT -> signo_zodiacal_geminis | n=10
   Iter 25/100
   Iter 50/100
   Iter 75/100
   Iter 100/100

DT -> signo_zodiacal_leo | n=10
   Iter 25/100
   Iter 50/100
   Iter 75/100
   Iter 100/100

DT -> signo_zodiacal_libra | n=10
   Iter 25/100
   Iter 50/100
   Iter 75/100
   Iter 100/100

DT -> signo_zodiacal_piscis | n=10
   Iter 25/100
  

# Resultados Gráficos

In [1]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt


# =========================================================
# CONFIG
# =========================================================
CARPETA_IN = Path("../4_results/modelo_dt_sensibilidad_n")
CARPETA_FIG = CARPETA_IN / "figuras"
CARPETA_FIG.mkdir(parents=True, exist_ok=True)

RUTA_RESUMEN = CARPETA_IN / "dt_sensibilidad_resumen_global.csv"
RUTA_EJECUCION = CARPETA_IN / "dt_sensibilidad_ejecucion.csv"

N_ORIGINAL = 100


# =========================================================
# CARGA
# =========================================================
df_resumen = pd.read_csv(RUTA_RESUMEN)
df_ejecucion = pd.read_csv(RUTA_EJECUCION)

df_resumen = df_resumen.sort_values("n_sample")
df_ejecucion = df_ejecucion.sort_values("n_sample")


# =========================================================
# HELPERS: ETIQUETAS Y PUNTO ORIGINAL
# =========================================================
def etiquetar_puntos_alternados(x, y, formato="{:.3f}", offsets=(5, -8), fontsize=8):
    """
    Agrega etiquetas numéricas alternando una arriba y otra abajo
    para evitar superposición visual.
    """
    for i, (xi, yi) in enumerate(zip(x, y)):
        offset_y = offsets[i % 2]

        plt.annotate(
            formato.format(yi),
            (xi, yi),
            textcoords="offset points",
            xytext=(0, offset_y),
            ha="center",
            va="bottom" if offset_y > 0 else "top",
            fontsize=fontsize
        )


def etiquetar_puntos_derecha(x, y, formato="{:.3f}", offset_x=8, fontsize=8):
    """
    Agrega etiquetas numéricas a la derecha de cada punto del gráfico.
    """
    for xi, yi in zip(x, y):
        plt.annotate(
            formato.format(yi),
            (xi, yi),
            textcoords="offset points",
            xytext=(offset_x, 0),
            ha="left",
            va="center",
            fontsize=fontsize
        )


def resaltar_n_original(df, y_col, n_original=N_ORIGINAL):
    """
    Marca en rojo el punto correspondiente al tamaño de muestra original.
    """
    punto = df[df["n_sample"] == n_original]

    if not punto.empty:
        plt.scatter(
            punto["n_sample"],
            punto[y_col],
            color="red",
            s=80,
            zorder=5,
            label=f"n original = {n_original}"
        )


# =========================================================
# GRÁFICO 1: ROC-AUC promedio vs tamaño de muestra
# =========================================================
plt.figure(figsize=(8, 5))

plt.plot(
    df_resumen["n_sample"],
    df_resumen["cv_roc_auc_mean"],
    marker="o"
)

resaltar_n_original(df_resumen, "cv_roc_auc_mean")

plt.axhline(y=0.5, linestyle="--", linewidth=1)

etiquetar_puntos_alternados(
    df_resumen["n_sample"],
    df_resumen["cv_roc_auc_mean"],
    formato="{:.3f}",
    offsets=(5, -8),
    fontsize=8
)

plt.ylim(0.48, 0.52)
plt.xlabel("Tamaño de muestra en bootstrapping")
plt.ylabel("ROC-AUC promedio")
plt.title("ROC-AUC promedio según tamaño de muestra")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(CARPETA_FIG / "dt_auc_promedio_vs_n.png", dpi=300)
plt.close()


# =========================================================
# GRÁFICO 2: Desviación estándar del ROC-AUC vs tamaño de muestra
# =========================================================
plt.figure(figsize=(8, 5))

plt.plot(
    df_resumen["n_sample"],
    df_resumen["cv_roc_auc_std"],
    marker="o"
)

resaltar_n_original(df_resumen, "cv_roc_auc_std")

etiquetar_puntos_derecha(
    df_resumen["n_sample"],
    df_resumen["cv_roc_auc_std"],
    formato="{:.3f}",
    offset_x=8,
    fontsize=8
)

plt.xlabel("Tamaño de muestra en bootstrapping")
plt.ylabel("Desviación estándar del ROC-AUC")
plt.title("Variabilidad del ROC-AUC según tamaño de muestra")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(CARPETA_FIG / "dt_auc_std_vs_n.png", dpi=300)
plt.close()


# =========================================================
# GRÁFICO 3: Tiempo promedio por iteración vs tamaño de muestra
# =========================================================
plt.figure(figsize=(8, 5))

plt.plot(
    df_ejecucion["n_sample"],
    df_ejecucion["tiempo_promedio_iteracion_seg"],
    marker="o"
)

resaltar_n_original(df_ejecucion, "tiempo_promedio_iteracion_seg")

etiquetar_puntos_alternados(
    df_ejecucion["n_sample"],
    df_ejecucion["tiempo_promedio_iteracion_seg"],
    formato="{:.3f}",
    offsets=(5, -8),
    fontsize=8
)

plt.xlabel("Tamaño de muestra en bootstrapping")
plt.ylabel("Tiempo promedio por iteración (segundos)")
plt.title("Tiempo promedio por iteración según tamaño de muestra")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(CARPETA_FIG / "dt_tiempo_promedio_iteracion_vs_n.png", dpi=300)
plt.close()


# =========================================================
# GRÁFICOS OPCIONALES
# =========================================================
GENERAR_GRAFICOS_OPCIONALES = True

if GENERAR_GRAFICOS_OPCIONALES:
    # -----------------------------------------------------
    # Gráfico opcional: Tiempo total vs tamaño de muestra
    # -----------------------------------------------------
    plt.figure(figsize=(8, 5))

    plt.plot(
        df_ejecucion["n_sample"],
        df_ejecucion["tiempo_total_seg"],
        marker="o"
    )

    resaltar_n_original(df_ejecucion, "tiempo_total_seg")

    etiquetar_puntos_alternados(
        df_ejecucion["n_sample"],
        df_ejecucion["tiempo_total_seg"],
        formato="{:.2f}",
        offsets=(5, -8),
        fontsize=8
    )

    plt.xlabel("Tamaño de muestra en bootstrapping")
    plt.ylabel("Tiempo total de ejecución (segundos)")
    plt.title("Tiempo total de ejecución según tamaño de muestra")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(CARPETA_FIG / "dt_tiempo_total_vs_n.png", dpi=300)
    plt.close()

    # -----------------------------------------------------
    # Gráfico opcional: Memoria peak observada vs tamaño de muestra
    # -----------------------------------------------------
    plt.figure(figsize=(8, 5))

    plt.plot(
        df_ejecucion["n_sample"],
        df_ejecucion["memoria_peak_observada_mb"],
        marker="o"
    )

    resaltar_n_original(df_ejecucion, "memoria_peak_observada_mb")

    etiquetar_puntos_alternados(
        df_ejecucion["n_sample"],
        df_ejecucion["memoria_peak_observada_mb"],
        formato="{:.1f}",
        offsets=(5, -8),
        fontsize=8
    )

    plt.xlabel("Tamaño de muestra en bootstrapping")
    plt.ylabel("Memoria peak observada (MB)")
    plt.title("Memoria observada según tamaño de muestra")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(CARPETA_FIG / "dt_memoria_peak_vs_n.png", dpi=300)
    plt.close()


# =========================================================
# TABLA RESUMEN PARA TESIS
# =========================================================
df_tabla = df_resumen[[
    "n_sample",
    "cv_accuracy_mean",
    "cv_accuracy_std",
    "cv_f1_mean",
    "cv_f1_std",
    "cv_roc_auc_mean",
    "cv_roc_auc_std",
    "cv_pr_auc_mean",
    "cv_pr_auc_std"
]].merge(
    df_ejecucion[[
        "n_sample",
        "tiempo_total_seg",
        "tiempo_promedio_iteracion_seg",
        "memoria_peak_observada_mb"
    ]],
    on="n_sample",
    how="left"
)

# Redondeo para que sea más legible al llevarlo a LaTeX o Excel
cols_redondear = [
    "cv_accuracy_mean",
    "cv_accuracy_std",
    "cv_f1_mean",
    "cv_f1_std",
    "cv_roc_auc_mean",
    "cv_roc_auc_std",
    "cv_pr_auc_mean",
    "cv_pr_auc_std",
    "tiempo_total_seg",
    "tiempo_promedio_iteracion_seg",
    "memoria_peak_observada_mb"
]

for col in cols_redondear:
    if col in df_tabla.columns:
        df_tabla[col] = df_tabla[col].round(4)

df_tabla.to_csv(CARPETA_IN / "dt_sensibilidad_tabla_tesis.csv", index=False)

print("Gráficos principales generados en:", CARPETA_FIG)
print(" - dt_auc_promedio_vs_n.png")
print(" - dt_auc_std_vs_n.png")
print(" - dt_tiempo_promedio_iteracion_vs_n.png")

if GENERAR_GRAFICOS_OPCIONALES:
    print("Gráficos opcionales generados:")
    print(" - dt_tiempo_total_vs_n.png")
    print(" - dt_memoria_peak_vs_n.png")

print("Tabla para tesis:", CARPETA_IN / "dt_sensibilidad_tabla_tesis.csv")

Gráficos principales generados en: ..\4_results\modelo_dt_sensibilidad_n\figuras
 - dt_auc_promedio_vs_n.png
 - dt_auc_std_vs_n.png
 - dt_tiempo_promedio_iteracion_vs_n.png
Gráficos opcionales generados:
 - dt_tiempo_total_vs_n.png
 - dt_memoria_peak_vs_n.png
Tabla para tesis: ..\4_results\modelo_dt_sensibilidad_n\dt_sensibilidad_tabla_tesis.csv
